<a href="https://colab.research.google.com/github/saathvikpd/AmazonReviewsDataAnalysis/blob/helpfulness-analysis-(all-beauty-only)/ECE_143.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import pandas as pd

def read_jsonl_stream(path, usecols=None, max_rows=None):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows:
                break
            obj = json.loads(line)
            if usecols:
                obj = {k: obj.get(k) for k in usecols}
            rows.append(obj)
    return pd.DataFrame(rows)

In [ ]:
review_cols = [
    "parent_asin",
    "rating",
    "title",
    "text",
    "verified_purchase",
    "helpful_vote",
    "timestamp"
]

reviews = read_jsonl_stream(
    "/content/drive/MyDrive/Amazon Review/All Beauty/All_Beauty.jsonl",
    usecols=review_cols,
    max_rows=200000
)

print(reviews.shape)
reviews.head()

(200000, 7)


,parent_asin,rating,title,text,verified_purchase,helpful_vote,timestamp
0,B00YQ6X8EO,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,True,0,1588687728923
1,B081TJ8YS3,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",True,1,1588615855070
2,B097R46CSY,5.0,Yes!,"Smells good, feels great!",True,2,1589665266052
3,B09JS339BZ,1.0,Synthetic feeling,Felt synthetic,True,0,1643393630220
4,B08BZ63GMJ,5.0,A+,Love it,True,0,1609322563534


In [ ]:
reviews["text"] = reviews["text"].fillna("")
reviews["title"] = reviews["title"].fillna("")

reviews["full_text"] = (reviews["title"] + " " + reviews["text"]).str.strip()

reviews["n_words"] = reviews["full_text"].str.split().str.len()
reviews["n_chars"] = reviews["full_text"].str.len()

reviews["helpful_vote"] = pd.to_numeric(reviews["helpful_vote"], errors="coerce").fillna(0).astype(int)
reviews["helpful_any"] = (reviews["helpful_vote"] > 0).astype(int)

reviews["super_short"] = reviews["n_words"] <= 3

reviews["extreme"] = reviews["rating"].isin([1.0, 5.0])

In [ ]:
meta_cols = [
    "parent_asin",
    "price",
    "main_category",
    "average_rating",
    "rating_number",
    "categories"
]

meta = read_jsonl_stream(
    "/content/drive/MyDrive/Amazon Review/All Beauty/meta_All_Beauty.jsonl",
    usecols=meta_cols
)

meta["price"] = pd.to_numeric(meta["price"], errors="coerce")
meta.head()

,parent_asin,price,main_category,average_rating,rating_number,categories
0,B01CUPMQZE,NaN,All Beauty,4.8,10,[]
1,B076WQZGPM,NaN,All Beauty,4.5,3,[]
2,B000B658RI,NaN,All Beauty,4.4,26,[]
3,B088FKY3VD,NaN,All Beauty,3.1,102,[]
4,B07NGFDN6G,NaN,All Beauty,4.3,7,[]


In [ ]:
df = reviews.merge(meta, how="left", on="parent_asin")
df.shape

(200000, 18)

In [ ]:
#Test 1: Extreme Ratings

df.groupby("rating")["helpful_any"].mean()

,helpful_any
rating,
1.0,0.314417
2.0,0.276384
3.0,0.269987
4.0,0.273395
5.0,0.267080


In [ ]:
#Test 2: Length Diminishing Returns

bins = [0,3,20,50,100,200,400,10000]
df["length_bin"] = pd.cut(df["n_words"], bins=bins)

df.groupby("length_bin")["helpful_any"].mean()

/tmp/ipython-input-273/584896248.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("length_bin")["helpful_any"].mean()


,helpful_any
length_bin,
"(0, 3]",0.099272
"(3, 20]",0.169826
"(20, 50]",0.283103
"(50, 100]",0.413571
"(100, 200]",0.517756
"(200, 400]",0.599204
"(400, 10000]",0.735521


In [ ]:
#Test 3: Super short reviews (word < 3)

df.groupby("super_short")["helpful_any"].mean()

,helpful_any
super_short,
False,0.279326
True,0.099272


In [ ]:
# Test 4 Verified Purchase Effect

df.groupby("verified_purchase")["helpful_any"].mean()

,helpful_any
verified_purchase,
False,0.272739
True,0.274307


# Research Questions

In [ ]:
product_counts = df.groupby("parent_asin")["parent_asin"].transform("count")
df["product_review_count"] = product_counts

In [ ]:
import statsmodels.formula.api as smf

model = smf.logit(
    "helpful_any ~ extreme + n_words + verified_purchase + product_review_count",
    data=df
).fit()

print(model.summary())

Optimization terminated successfully.
         Current function value: 0.555029
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:            helpful_any   No. Observations:               200000
Model:                          Logit   Df Residuals:                   199995
Method:                           MLE   Df Model:                            4
Date:                Thu, 26 Feb 2026   Pseudo R-squ.:                 0.05499
Time:                        19:04:32   Log-Likelihood:            -1.1101e+05
converged:                       True   LL-Null:                   -1.1746e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                    -2.2490      0.022   -100.044      0.000      -2.

In [ ]:
df["price_bucket"] = pd.qcut(df["price"], q=3, labels=["Low","Mid","High"])
df.groupby("price_bucket")["helpful_any"].mean()

/tmp/ipython-input-273/2971918667.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("price_bucket")["helpful_any"].mean()


,helpful_any
price_bucket,
Low,0.266287
Mid,0.306358
High,0.356943


In [ ]:
model2 = smf.logit(
    "helpful_any ~ extreme * price_bucket + n_words + verified_purchase",
    data=df
).fit()

print(model2.summary())